In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("data/df_limpio.csv")
df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,6B4H6E,22,16000.0,10000,769,22.0,1,8.96,48.0,0.32,Escolar,Desempleado,Soltero,1.0,1.0,Vivienda,0.0,0,65.85
1,VKTUXP,33,38308.0,16474,699,70.0,3,14.07,36.0,0.20,Máster,Jornada completa,Casado,1.0,1.0,Vivienda,0.0,0,260.60
2,5P9OVL,51,51271.0,40770,812,309.0,1,12.66,12.0,0.36,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,1.0,0,221.02
3,YL0W8G,30,31923.0,36707,813,43.0,1,8.96,24.0,0.32,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,0.0,0,125.00
4,YP8WTV,28,27373.0,86981,765,60.0,2,10.07,48.0,0.55,Escolar,Jornada completa,Soltero,0.0,1.0,Vivienda,0.0,0,800.00


In [3]:
df2=pd.read_csv("data/resultado_recomendaciones.csv",sep=";")
df2.head()

,ID,Recomendado,Recomendado_Final,Saldo_Impago_Mitad
0,6B4H6E,Aleman,Aleman,5000.00
1,VKTUXP,Aleman,Aleman,8237.00
2,5P9OVL,Frances,Frances (ampliar plazo),21026.59
3,YL0W8G,Frances,Frances (ampliar plazo),19172.14
4,YP8WTV,Frances,Denegar / estudio especial,47837.16


Poner las columnas necesarias al mes porque los créditos se pagan una vez

In [4]:
df3=pd.read_csv('data/cuadros_amort.csv')

In [5]:
df3.head()

,Unnamed: 0,ID,Mes,Fr_Cuota,Fr_Intereses,Fr_Capital_Amortizado,Fr_Capital_Pendiente,Fr_Flujo_Caja,Al_Cuota,Al_Intereses,Al_Capital_Amortizado,Al_Capital_Pendiente,Al_Flujo_Caja
0,0,6B4H6E,1,248.66,74.67,173.99,9826.01,-173.99,283.00,74.67,208.33,9791.67,-208.33
1,1,6B4H6E,2,248.66,73.37,175.29,9650.71,-175.29,281.44,73.11,208.33,9583.33,-208.33
2,2,6B4H6E,3,248.66,72.06,176.60,9474.11,-176.60,279.89,71.56,208.33,9375.00,-208.33
3,3,6B4H6E,4,248.66,70.74,177.92,9296.19,-177.92,278.33,70.00,208.33,9166.67,-208.33
4,4,6B4H6E,5,248.66,69.41,179.25,9116.94,-179.25,276.78,68.44,208.33,8958.33,-208.33


In [6]:
df3.columns

Index(['Unnamed: 0', 'ID', 'Mes', 'Fr_Cuota', 'Fr_Intereses',
       'Fr_Capital_Amortizado', 'Fr_Capital_Pendiente', 'Fr_Flujo_Caja',
       'Al_Cuota', 'Al_Intereses', 'Al_Capital_Amortizado',
       'Al_Capital_Pendiente', 'Al_Flujo_Caja'],
      dtype='object')

In [7]:
df["Ratio_Interes_Mensual"] = df["Ratio_Interes"]/100/12

Merge de los datasets que necesitamos para resolver este objetivo: calcular el valor actual y el final de cada préstamo, dependiendo si son rentas de tipo constante (francés) o variable (alemán)

In [8]:
# Sistema francés: cuota constante
franc = df3.groupby("ID").agg(
    n=("Mes", "max"),
    cuota_fr=("Fr_Cuota", "first")      # par qu esirve
)

df = df.merge(franc, on="ID", how="left")

In [9]:
alem = df3.merge(
    df[["ID", "Ratio_Interes_Mensual", "n"]],
    on="ID",
    how="left"
)

In [10]:
# Incorporar la amortización recomendada a los resultados
df = df.merge(
    df2[["ID", "Recomendado"]].drop_duplicates("ID"),
    on="ID",
    how="left"
)

# Comprobar qué valores aparecen en la columna recomendado
print(df["Recomendado"].value_counts())

Recomendado
Frances    81046
Aleman     20849
Name: count, dtype: int64


In [11]:
i = df["Ratio_Interes_Mensual"]
n = df["n"]     #periodo mensual
C = df["cuota_fr"]

En los seguros las rentas siempre son prepagables, asi que usamos esas fórmulas

## SISTEMA FRÁNCES

In [12]:
# Valor actual francés
df["valor_actual_frances"] = np.where(i != 0,C * (1 - (1 + i) ** (-n)) / i * (1 + i),C * n)

# Valor final francés
df["valor_final_frances"] = np.where(i != 0,C * (((1 + i) ** n - 1) / i) * (1 + i),C * n)

## SISTEMA ALEMÁN

In [13]:
alem["valor_actual_aleman"] = (alem["Al_Cuota"] /(1 + alem["Ratio_Interes_Mensual"]) ** (alem["Mes"] - 1))

alem["valor_final_aleman"] = (alem["Al_Cuota"] *(1 + alem["Ratio_Interes_Mensual"]) **(alem["n"] - alem["Mes"] + 1))

In [14]:
resumen_aleman = alem.groupby("ID").agg(
    valor_actual_aleman=("valor_actual_aleman", "sum"),
    valor_final_aleman=("valor_final_aleman", "sum")
).reset_index()

In [15]:
df = df.merge(resumen_aleman, on="ID", how="left")

In [16]:
# Seleccionar el resultado según la recomendación
recomendado = df["Recomendado"].astype("string").str.lower()

es_frances = recomendado.str.contains("franc")
es_aleman = recomendado.str.contains("alem")

In [17]:
df["valor_actual_recomendado"] = np.select(
    [es_frances, es_aleman],
    [df["valor_actual_frances"], df["valor_actual_aleman"]])

df["valor_final_recomendado"] = np.select(
    [es_frances, es_aleman],
    [df["valor_final_frances"], df["valor_final_aleman"]])

In [18]:
# según el sistema elegido se ha obtenido el Valor Actual y Final  
valor_ac_fin=df[[
    "ID",
    "Recomendado",
    "valor_actual_recomendado",
    "valor_final_recomendado",
    "n"]]

valor_ac_fin.head()


,ID,Recomendado,valor_actual_recomendado,valor_final_recomendado,n
0,6B4H6E,Aleman,10074.663240,14398.042770,48
1,VKTUXP,Aleman,16667.159762,25357.762980,36
2,5P9OVL,Frances,41200.179544,46729.677686,12
3,YL0W8G,Frances,36981.161855,44209.643422,24
4,YP8WTV,Frances,87710.992285,130995.968175,48


In [19]:
valor_ac_fin.isna().sum()

ID                          0
Recomendado                 0
valor_actual_recomendado    0
valor_final_recomendado     0
n                           0
dtype: int64

In [20]:
valor_ac_fin.shape

(101895, 5)